# Local LLM Inference on L4 GPUs

This notebook demonstrates how to run Large Language Models locally on L4 GPUs for inference tasks. We'll cover model loading, memory optimization, and building interactive applications.

## Learning Objectives

* Load and run LLMs on L4 GPUs
* Implement memory optimization techniques
* Build interactive chat interfaces
* Optimize inference performance
* Handle different model architectures

## Setup and Imports

Let's import the necessary libraries for LLM inference.

In [ ]:
import torch
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    pipeline, TextStreamer, TextIteratorStreamer
)
import accelerate
import bitsandbytes
import numpy as np
import time
import gc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

## Model Selection for L4 GPUs

L4 GPUs have 24GB of VRAM, which is perfect for:
* 7B parameter models (full precision)
* 13B parameter models (with quantization)
* Larger models with 4-bit quantization

Let's start with a 7B model that fits comfortably in L4 memory.

In [ ]:
# Model configuration for L4 GPU
MODEL_CONFIGS = {
    "llama-7b": {
        "model_name": "meta-llama/Llama-2-7b-chat-hf",
        "memory_gb": 14,
        "description": "Llama 2 7B Chat - Good balance of performance and memory"
    },
    "mistral-7b": {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.1",
        "memory_gb": 14,
        "description": "Mistral 7B Instruct - Fast and efficient"
    },
    "codellama-7b": {
        "model_name": "codellama/CodeLlama-7b-Instruct-hf",
        "memory_gb": 14,
        "description": "Code Llama 7B - Specialized for code generation"
    },
    "phi-2": {
        "model_name": "microsoft/phi-2",
        "memory_gb": 5,
        "description": "Phi-2 2.7B - Very efficient, good for experimentation"
    }
}

print("Available models for L4 GPU:")
for name, config in MODEL_CONFIGS.items():
    print(f"  {name}: {config['description']} ({config['memory_gb']}GB)")

## Memory Optimization Techniques

Let's implement several memory optimization techniques to maximize our GPU utilization.

In [ ]:
def get_gpu_memory_info():
    """Get current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        print(f"GPU Memory Usage:")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved: {reserved:.2f} GB")
        print(f"  Total: {total:.2f} GB")
        print(f"  Available: {total - reserved:.2f} GB")
        return allocated, reserved, total
    return 0, 0, 0

def clear_gpu_memory():
    """Clear GPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("GPU memory cleared")

def load_model_with_optimization(model_name, use_quantization=True, use_flash_attention=True):
    """Load model with memory optimizations"""
    print(f"Loading {model_name}...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Configure model loading options
    model_kwargs = {
        "torch_dtype": torch.float16,  # Use half precision
        "device_map": "auto",  # Automatic device mapping
        "low_cpu_mem_usage": True,  # Reduce CPU memory usage
    }
    
    # Add quantization if requested
    if use_quantization:
        model_kwargs["load_in_8bit"] = True
        print("Using 8-bit quantization")
    
    # Add flash attention if available
    if use_flash_attention:
        try:
            model_kwargs["attn_implementation"] = "flash_attention_2"
            print("Using Flash Attention 2")
        except:
            print("Flash Attention 2 not available, using default")
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    
    print(f"Model loaded successfully!")
    get_gpu_memory_info()
    
    return model, tokenizer

# Check initial memory
print("Initial GPU memory:")
get_gpu_memory_info()

## Loading Our First Model

Let's start with Phi-2, a smaller but capable model that's perfect for learning.

In [ ]:
# Load Phi-2 model (smaller, good for learning)
model_name = MODEL_CONFIGS["phi-2"]["model_name"]
model, tokenizer = load_model_with_optimization(model_name, use_quantization=False)

print(f"\nModel: {model_name}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Basic Text Generation

Let's start with basic text generation to understand how the model works.

In [ ]:
def generate_text(prompt, max_length=100, temperature=0.7, top_p=0.9):
    """Generate text from a prompt"""
    print(f"Prompt: {prompt}")
    print("Generated text:")
    print("-" * 50)
    
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generation_time = time.time() - start_time
    
    # Decode output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the new text
    new_text = generated_text[len(prompt):]
    
    print(new_text)
    print("-" * 50)
    print(f"Generation time: {generation_time:.2f} seconds")
    print(f"Tokens generated: {len(outputs[0]) - len(inputs['input_ids'][0])}")
    print(f"Tokens per second: {(len(outputs[0]) - len(inputs['input_ids'][0])) / generation_time:.1f}")
    
    return new_text

# Test basic generation
prompt = "The future of artificial intelligence is"
generate_text(prompt, max_length=50)

## Interactive Chat Interface

Let's build a simple chat interface to interact with our model.

In [ ]:
class SimpleChatBot:
    def __init__(self, model, tokenizer, system_prompt="You are a helpful AI assistant."):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.conversation_history = []
        
    def chat(self, user_input, max_length=150, temperature=0.7):
        """Process user input and generate response"""
        # Add user message to history
        self.conversation_history.append(f"Human: {user_input}")
        
        # Build conversation context
        context = self.system_prompt + "\n\n"
        context += "\n".join(self.conversation_history[-6:])  # Keep last 6 exchanges
        context += "\nAssistant:"
        
        # Generate response
        response = generate_text(context, max_length=max_length, temperature=temperature)
        
        # Add assistant response to history
        self.conversation_history.append(f"Assistant: {response.strip()}")
        
        return response.strip()
    
    def clear_history(self):
        """Clear conversation history"""
        self.conversation_history = []
        print("Conversation history cleared.")

# Create chatbot instance
chatbot = SimpleChatBot(model, tokenizer)

# Test the chatbot
print("🤖 Chatbot initialized! Let's have a conversation:")
print("Type 'quit' to exit, 'clear' to clear history")
print("=" * 60)

# Example conversation
test_questions = [
    "Hello! Can you help me understand machine learning?",
    "What are the benefits of using GPUs for AI?",
    "How can I optimize memory usage when training large models?"
]

for question in test_questions:
    print(f"\n👤 Human: {question}")
    response = chatbot.chat(question)
    print(f"🤖 Assistant: {response}")
    print("-" * 40)

## Batch Processing for Efficiency

Processing multiple requests in batches can significantly improve throughput.

In [ ]:
def batch_generate(prompts, max_length=100, batch_size=4):
    """Generate text for multiple prompts in batches"""
    results = []
    
    print(f"Processing {len(prompts)} prompts in batches of {batch_size}...")
    
    start_time = time.time()
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        
        # Generate for batch
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode batch results
        for j, output in enumerate(outputs):
            generated_text = tokenizer.decode(output, skip_special_tokens=True)
            new_text = generated_text[len(batch_prompts[j]):]
            results.append(new_text.strip())
        
        print(f"Processed batch {i//batch_size + 1}/{(len(prompts) + batch_size - 1)//batch_size}")
    
    total_time = time.time() - start_time
    print(f"\nBatch processing completed in {total_time:.2f} seconds")
    print(f"Average time per prompt: {total_time/len(prompts):.2f} seconds")
    
    return results

# Test batch processing
test_prompts = [
    "Explain the concept of",
    "The benefits of using",
    "How does machine learning",
    "What is the difference between",
    "Describe the process of",
    "Why is it important to"
]

batch_results = batch_generate(test_prompts, max_length=30, batch_size=3)

print("\nResults:")
for i, (prompt, result) in enumerate(zip(test_prompts, batch_results)):
    print(f"{i+1}. {prompt}... -> {result}")

## Performance Monitoring

Let's monitor our model's performance and resource usage.

In [ ]:
def benchmark_model(prompts, iterations=5):
    """Benchmark model performance"""
    print("🚀 Running performance benchmark...")
    
    times = []
    tokens_per_second = []
    
    for i in range(iterations):
        prompt = prompts[i % len(prompts)]
        
        # Time generation
        start_time = time.time()
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generation_time = time.time() - start_time
        tokens_generated = len(outputs[0]) - len(inputs['input_ids'][0])
        
        times.append(generation_time)
        tokens_per_second.append(tokens_generated / generation_time)
        
        print(f"Iteration {i+1}: {generation_time:.2f}s, {tokens_generated} tokens, {tokens_generated/generation_time:.1f} tok/s")
    
    # Calculate statistics
    avg_time = np.mean(times)
    avg_tokens_per_sec = np.mean(tokens_per_second)
    
    print(f"\n📊 Benchmark Results:")
    print(f"Average generation time: {avg_time:.2f} seconds")
    print(f"Average tokens per second: {avg_tokens_per_sec:.1f}")
    print(f"Time std deviation: {np.std(times):.2f} seconds")
    
    # Memory usage
    get_gpu_memory_info()
    
    return {
        'avg_time': avg_time,
        'avg_tokens_per_sec': avg_tokens_per_sec,
        'times': times,
        'tokens_per_second': tokens_per_second
    }

# Run benchmark
benchmark_prompts = [
    "The future of AI is",
    "Machine learning helps us",
    "Deep learning models can",
    "Neural networks are",
    "Artificial intelligence will"
]

benchmark_results = benchmark_model(benchmark_prompts, iterations=3)

## Loading a Larger Model (Optional)

If you have enough memory, let's try loading a larger model like Mistral-7B.

In [ ]:
# Uncomment to load Mistral-7B (requires ~14GB VRAM)
# Clear current model first
clear_gpu_memory()

# Check available memory
allocated, reserved, total = get_gpu_memory_info()
available_memory = total - reserved

if available_memory > 15:  # Need at least 15GB free
    print("🔄 Loading Mistral-7B model...")
    mistral_model, mistral_tokenizer = load_model_with_optimization(
        MODEL_CONFIGS["mistral-7b"]["model_name"], 
        use_quantization=True  # Use quantization to fit in memory
    )
    
    # Test Mistral
    print("\n🧪 Testing Mistral-7B:")
    mistral_prompt = "Write a short story about a robot learning to paint."
    mistral_response = generate_text(mistral_prompt, max_length=100)
    
else:
    print(f"❌ Not enough memory for Mistral-7B. Available: {available_memory:.1f}GB, Required: ~15GB")
    print("💡 Try clearing memory or use a smaller model.")

## Cleanup and Memory Management

Always clean up GPU memory when you're done with models.

In [ ]:
def cleanup_models():
    """Clean up all loaded models"""
    global model, tokenizer
    
    # Delete model references
    if 'model' in globals():
        del model
    if 'tokenizer' in globals():
        del tokenizer
    
    # Clear GPU memory
    clear_gpu_memory()
    
    print("✅ Models cleaned up successfully")
    get_gpu_memory_info()

# Cleanup (uncomment when ready)
# cleanup_models()

print("💡 To clean up models and free GPU memory, uncomment the cleanup_models() call above")

## Exercises

### Basic Exercises
1. **Model Comparison**: Load different models (Phi-2, Mistral-7B) and compare their:
   - Response quality
   - Generation speed
   - Memory usage

2. **Parameter Tuning**: Experiment with different generation parameters:
   - Temperature (0.1 to 1.0)
   - Top-p (0.1 to 1.0)
   - Max length (50 to 200)

3. **Memory Optimization**: Try different optimization techniques:
   - 8-bit vs 4-bit quantization
   - Flash Attention vs standard attention
   - Batch size optimization

### Advanced Exercises
4. **Custom Chatbot**: Build a specialized chatbot for:
   - Technical support
   - Code review
   - Creative writing

5. **Performance Analysis**: Create a comprehensive benchmark that measures:
   - Throughput (tokens/second)
   - Latency (time to first token)
   - Memory efficiency
   - Quality metrics

6. **Error Handling**: Implement robust error handling for:
   - Out of memory errors
   - Invalid inputs
   - Model loading failures

---
**Next notebook:** [Fine-tuning Models on HPC](03_fine_tuning.ipynb)

// ...at the top of each notebook...
{
 "cell_type": "markdown",
 "metadata": {},
 "source": [
  "© mattbixley"
 ]
},